# 🧪 Lab 5: Gemini 로드밸런싱 테스트

## 테스트 목표

APIM 뒤에 Gemini 백엔드 3개(API Key 3개)를 연결하고,
**Gemini 네이티브 포맷**으로 직접 호출하여 백엔드 풀이 정상 동작하는지 검증합니다.

| # | 테스트 | 기대 결과 |
|---|---|---|
| 1 | 단일 Gemini 호출 | 200 + Gemini 네이티브 응답 |
| 2 | 응답 포맷 검증 | `candidates`, `usageMetadata` 필드 존재 |
| 3 | 로드밸런싱 (12회 호출) | 모두 200 성공 (3개 백엔드 분배) |

### 사전 준비

Lab 5 README의 1~5단계를 완료해야 합니다:
1. Named Values 3개 등록 (gemini-api-key-1, 2, 3)
2. Gemini 백엔드 3개 생성 (credentials에 API Key 포함)
3. 백엔드 풀 생성 (gemini-backend-pool)
4. Gemini API 등록 (Operation: POST /models/gemini-2.5-flash-lite:generateContent)
5. 정책 적용 (set-backend-service 한 줄)

In [28]:
import os, json, time
import requests
from dotenv import load_dotenv

load_dotenv("../../.env", override=True)

APIM_URL = os.getenv("APIM_URL")
APIM_KEY = os.getenv("APIM_SUBSCRIPTION_KEY")

assert APIM_URL, "❌ APIM_URL이 설정되지 않았습니다."
assert APIM_KEY, "❌ APIM_SUBSCRIPTION_KEY가 설정되지 않았습니다."

GEMINI_URL = f"{APIM_URL}/gemini/models/gemini-2.5-flash-lite:generateContent"

def call_gemini(prompt="Hello!", max_tokens=256):
    """APIM Gemini API 호출 — Gemini 네이티브 포맷"""
    return requests.post(
        GEMINI_URL,
        headers={
            "Content-Type": "application/json",
            "Ocp-Apim-Subscription-Key": APIM_KEY,
        },
        json={
            "contents": [{"role": "user", "parts": [{"text": prompt}]}],
            "generationConfig": {"maxOutputTokens": max_tokens},
        },
        timeout=30,
    )

print("✅ 환경 설정 완료")
print(f"   APIM URL: {APIM_URL}")
print(f"   Gemini Endpoint: {GEMINI_URL}")

✅ 환경 설정 완료
   APIM URL: https://apim-ai-gw-aigateway-20260317.azure-api.net
   Gemini Endpoint: https://apim-ai-gw-aigateway-20260317.azure-api.net/gemini/models/gemini-2.5-flash-lite:generateContent


---
## 테스트 1: 단일 Gemini 호출

APIM → Gemini 백엔드 풀 경로가 정상 동작하는지 확인합니다.
Gemini 네이티브 포맷으로 요청하고, 네이티브 응답을 그대로 받습니다.

In [22]:
print("▶ 테스트 1: 단일 Gemini 호출\n")

resp = call_gemini(prompt="Say hello in Korean", max_tokens=256)

print(f"  HTTP Status: {resp.status_code}")

if resp.status_code == 200:
    data = resp.json()
    # Gemini 2.5 Flash는 thinking 모델 — parts에 thought와 text가 섞여 있을 수 있음
    parts = data.get("candidates", [{}])[0].get("content", {}).get("parts", [])
    text = next((p["text"] for p in parts if "text" in p and "thought" not in p), 
                parts[0].get("text", "N/A") if parts else "N/A")
    usage = data.get("usageMetadata", {})
    print(f"  응답: {text}")
    print(f"  토큰: prompt={usage.get('promptTokenCount', '?')}, "
          f"completion={usage.get('candidatesTokenCount', '?')}, "
          f"thinking={usage.get('thoughtsTokenCount', '?')}, "
          f"total={usage.get('totalTokenCount', '?')}")
    print(f"\n  ✅ Gemini 정상 응답 (네이티브 포맷)")
else:
    print(f"  ❌ 실패: {resp.text[:300]}")
    print(f"  → Lab 5 README 1~5단계를 확인하세요.")

▶ 테스트 1: 단일 Gemini 호출

  HTTP Status: 200
  응답: 안녕하세요! (Annyeonghaseyo!)

That's the most common and versatile way to say "hello" in Korean.
  토큰: prompt=5, completion=27, thinking=28, total=60

  ✅ Gemini 정상 응답 (네이티브 포맷)


---
## 테스트 2: 응답 포맷 검증

Gemini 네이티브 응답의 표준 필드가 모두 존재하는지 확인합니다.

In [23]:
print("▶ 테스트 2: 응답 포맷 검증\n")

if resp.status_code != 200:
    print("  ⚠️ 테스트 1이 실패하여 포맷 검증을 건너뜁니다.")
else:
    data = resp.json()
    parts = data.get("candidates", [{}])[0].get("content", {}).get("parts", [])
    has_text = any("text" in p for p in parts)

    fields = {
        "candidates (array)": isinstance(data.get("candidates"), list),
        "candidates[0].content.parts": len(parts) > 0,
        "parts에 text 포함": has_text,
        "candidates[0].finishReason": bool(data.get("candidates", [{}])[0].get("finishReason")),
        "usageMetadata.totalTokenCount": isinstance(data.get("usageMetadata", {}).get("totalTokenCount"), int),
    }

    all_ok = True
    for field, ok in fields.items():
        status = "✅" if ok else "❌"
        print(f"  {status} {field}")
        if not ok:
            all_ok = False

    print()
    if all_ok:
        print("  ✅ Gemini 네이티브 응답 필드 모두 정상!")
    else:
        print("  ⚠️ 일부 필드가 누락되었습니다. maxOutputTokens를 늘려보세요.")

▶ 테스트 2: 응답 포맷 검증

  ✅ candidates (array)
  ✅ candidates[0].content.parts
  ✅ parts에 text 포함
  ✅ candidates[0].finishReason
  ✅ usageMetadata.totalTokenCount

  ✅ Gemini 네이티브 응답 필드 모두 정상!


---
## 테스트 3: 로드밸런싱 검증 (12회 호출)

12회 연속 호출하여 백엔드 풀이 정상 동작하는지 확인합니다.
API Key는 백엔드 `credentials`에 설정되어 있으므로 APIM이 자동으로 분배합니다.

In [27]:
from collections import Counter

print("▶ 테스트 3: 로드밸런싱 검증 (12회 호출)\n")

NUM_REQUESTS = 12
results = []

print(f"  {'#':>3}  {'Status':>6}  Backend")
print(f"  {'─' * 50}")

for i in range(1, NUM_REQUESTS + 1):
    try:
        r = call_gemini(prompt=f"Say number {i}", max_tokens=100)
        status = r.status_code
        backend = r.headers.get("x-backend-id", "<missing>")
        results.append({"request": i, "status": status, "backend": backend})
        icon = "✅" if status == 200 else "❌"
        print(f"  {i:3d}  {icon} {status}  {backend}")
    except Exception as e:
        results.append({"request": i, "status": 0, "backend": "ERR"})
        print(f"  {i:3d}  ❌   0  에러 - {e}")
    time.sleep(1)

# 통계
success_count = sum(1 for r in results if r["status"] == 200)
fail_count = NUM_REQUESTS - success_count

print(f"\n{'─' * 50}")
print(f"  성공: {success_count}/{NUM_REQUESTS}  |  실패: {fail_count}/{NUM_REQUESTS}")

# 백엔드 분배
backend_count = Counter(r["backend"] for r in results if r["status"] == 200)
print(f"\n  {'백엔드':<30} {'요청수':>4} {'비율':>6}")
print(f"  {'─' * 45}")
for backend, count in backend_count.most_common():
    pct = count / success_count * 100 if success_count > 0 else 0
    bar = "█" * int(pct / 5)
    print(f"  {backend:<30} {count:>4} {pct:>5.0f}% {bar}")

print(f"\n  사용된 백엔드: {len(backend_count)}개")

if "<missing>" in backend_count:
    print("\n  ⚠️ x-backend-id 헤더가 없습니다. 5단계 outbound 정책을 Portal에 적용했는지 확인하세요.")

▶ 테스트 3: 로드밸런싱 검증 (12회 호출)

    #  Status  Backend
  ──────────────────────────────────────────────────
    1  ❌ 429  gemini-backend-3
    2  ❌ 429  gemini-backend-1
    3  ✅ 200  gemini-backend-2
    4  ❌ 429  gemini-backend-3
    5  ❌ 429  gemini-backend-1
    6  ❌ 429  gemini-backend-2
    7  ❌ 429  gemini-backend-3
    8  ❌ 429  gemini-backend-1
    9  ✅ 200  gemini-backend-2
   10  ❌ 429  gemini-backend-3
   11  ❌ 429  gemini-backend-1
   12  ❌ 429  gemini-backend-2

──────────────────────────────────────────────────
  성공: 2/12  |  실패: 10/12

  백엔드                             요청수     비율
  ─────────────────────────────────────────────
  gemini-backend-2                  2   100% ████████████████████

  사용된 백엔드: 1개


---
## 결과 요약

In [25]:
print("═" * 55)
print("  Gemini 로드밸런싱 테스트 요약")
print("═" * 55)
print()

# 테스트 1: 단일 호출
t1_ok = resp.status_code == 200
print(f"  테스트 1 (단일 호출):     {'✅ 정상' if t1_ok else '❌ 실패'}")

# 테스트 2: 포맷 검증
if t1_ok:
    d = resp.json()
    t2_ok = all([
        isinstance(d.get("candidates"), list),
        bool(d.get("candidates", [{}])[0].get("content", {}).get("parts")),
        isinstance(d.get("usageMetadata", {}).get("totalTokenCount"), int),
    ])
else:
    t2_ok = False
print(f"  테스트 2 (포맷 검증):     {'✅ 정상' if t2_ok else '❌ 실패'}")

# 테스트 3: 로드밸런싱
success_rate = success_count / NUM_REQUESTS * 100
t3_ok = success_rate >= 80
print(f"  테스트 3 (로드밸런싱):    {'✅ 정상' if t3_ok else '⚠️ 확인 필요'}")
print(f"    - 성공률: {success_rate:.0f}% ({success_count}/{NUM_REQUESTS})")
print(f"    - 백엔드 분배: {dict(backend_count)}")

# 판정
print(f"\n{'─' * 55}")
checks = {
    "모든 요청 성공 (200)": all(r["status"] == 200 for r in results),
    "2개 이상 백엔드 분산": len(backend_count) >= 2,
    "3개 백엔드 균등 분산": len(backend_count) >= 3,
}
for check, passed in checks.items():
    icon = "✅" if passed else "❌"
    print(f"  {icon} {check}")

print()
if all(checks.values()):
    print("  🎉 Gemini 3개 Key 로드밸런싱이 정상 동작합니다!")
elif checks["모든 요청 성공 (200)"]:
    print("  ⚠️ 요청은 성공하나 분배가 불균등합니다. 백엔드 풀 설정을 확인하세요.")
else:
    print("  ⚠️ 일부 테스트가 실패했습니다. README를 참고하여 설정을 확인하세요.")

═══════════════════════════════════════════════════════
  Gemini 로드밸런싱 테스트 요약
═══════════════════════════════════════════════════════

  테스트 1 (단일 호출):     ✅ 정상
  테스트 2 (포맷 검증):     ✅ 정상
  테스트 3 (로드밸런싱):    ✅ 정상
    - 성공률: 100% (12/12)
    - 백엔드 분배: {'gemini-backend-3': 4, 'gemini-backend-1': 4, 'gemini-backend-2': 4}

───────────────────────────────────────────────────────
  ✅ 모든 요청 성공 (200)
  ✅ 2개 이상 백엔드 분산
  ✅ 3개 백엔드 균등 분산

  🎉 Gemini 3개 Key 로드밸런싱이 정상 동작합니다!
